In [1]:
import pandas as pd
df = pd.read_json("../data/results/codetrans_m1_results.jsonl", lines=True)

In [2]:
df.dropna(inplace=True)

In [3]:
import re

def extract_code(text: str) -> str:
    # Pattern explanation:
    # ```(?:\w+)? -> Matches opening backticks and optional language name (non-capturing)
    # \s* -> Matches optional whitespace/newline after the language tag
    # (.*?)        -> Captures the actual code (non-greedy)
    # \s*```       -> Matches optional whitespace and closing backticks
    pattern = r"```(?:\w+)?\s*(.*?)\s*```"
    
    # re.DOTALL allows the '.' to match newline characters
    match = re.search(pattern, text, re.DOTALL)
    
    if match:
        return match.group(1).strip()
    
    # If no code block is found, return the original text
    return text

In [4]:
from pygments.lexers import get_lexer_by_name
from pygments.token import Token

def compare_generic_tokens(code1: str, code2: str, lang:  str) -> bool:
    lexer = get_lexer_by_name(lang)

    def extract_logic(code):
        # Filter out Whitespace and Comments
        return [
            tok_value.strip() 
            for tok_type, tok_value in lexer.get_tokens(code)
            if tok_type not in Token.Text and tok_type not in Token.Comment
            and tok_value.strip() != ""
        ]

    return extract_logic(code1) == extract_logic(code2)

In [5]:
df['model_output'] = df['model_output'].apply(lambda x: extract_code(x))

In [6]:
from codebleu import calc_codebleu
import sacrebleu

def bleu_score(pred, ref):
  return sacrebleu.sentence_bleu(pred, [ref]).score

def codebleu_score(pred, ref, lang):
    res = calc_codebleu(
        [ref],
        [pred],
        lang
    )
    return res["codebleu"]

In [7]:
from rouge_score import rouge_scorer, scoring
scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)

def rouge_l_aggregated(preds, refs):
  aggregator = scoring.BootstrapAggregator()
  for ref, pred in zip(refs, preds):
    aggregator.add_scores(scorer.score(ref, pred))
  return aggregator.aggregate()["rougeL"].mid.fmeasure

In [8]:
df["match"] = df.apply(lambda row: compare_generic_tokens(row['cs_code'], row['model_output'], lang="c#"), axis=1)
df["bleu"] = df.apply(lambda x: bleu_score(x.model_output, x.cs_code) / 100.0, axis=1)
df["codebleu"] = df.apply(lambda x: codebleu_score(x.model_output, x.cs_code, lang="c_sharp"), axis=1)
df["rougeL"] = [rouge_l_aggregated([p], [r]) for p, r in zip(df.model_output, df.cs_code)]

In [9]:
import pandas as pd
from scipy import stats as scipy_stats

# 1. Clean data and set metrics
df['model_name'] = df['model_name'].str.split('/').str[-1]
metrics = ['match', 'bleu', 'codebleu', 'rougeL']

# 2. Calculate Mean and Std
agg_stats = df.groupby(['model_name', 'prompt_type'])[metrics].agg(['mean', 'std'])

# 3. Format the strings (mean ± std)
display_df = pd.DataFrame(index=agg_stats.index)
for m in metrics:
    display_df[m] = agg_stats.apply(lambda x: f"{x[m]['mean']:.3f} ± {x[m]['std']:.3f}", axis=1)

# 4. Significance Styling Logic
def style_significance(data):
    style_df = pd.DataFrame('', index=data.index, columns=data.columns)
    
    # Iterate through unique models in the MultiIndex level 0
    for model in data.index.get_level_values(0).unique():
        model_mask = df['model_name'] == model
        group1_raw = df[model_mask & (df['prompt_type'] == 'zero_shot_prompt')]
        group2_raw = df[model_mask & (df['prompt_type'] == 'over_supervised_prompt')]
        
        for m in metrics:
            if not group1_raw.empty and not group2_raw.empty:
                # Perform T-Test
                _, p_val = scipy_stats.ttest_ind(group1_raw[m], group2_raw[m], nan_policy='omit')
                
                if p_val < 0.05:
                    # Find which prompt had the higher mean
                    m1, m2 = group1_raw[m].mean(), group2_raw[m].mean()
                    # Map back to the actual prompt_type values in the index
                    better_prompt = 'zero_shot_prompt' if m1 > m2 else 'over_supervised_prompt'
                    style_df.loc[(model, better_prompt), m] = 'font-weight: bold'
    return style_df

# 5. Display with MultiIndex (this handles the cell merging)
styled_table = display_df.style.apply(style_significance, axis=None)

# Optional: Add borders to make the merging look even cleaner
styled_table.set_table_styles([
    {'selector': 'th', 'props': [('border', '1px solid lightgrey')]},
    {'selector': 'td', 'props': [('border', '1px solid lightgrey')]}
])

styled_table

In [10]:
df

,id,java_code,cs_code,model_name,prompt_type,model_output,match,bleu,codebleu,rougeL
0,1434,public int lastLength() {return lastLength;},public virtual int LastLength(){return lastLen...,gpt-oss-120b,zero_shot_prompt,public int LastLength()\n{\n return lastLen...,False,0.817613,0.433661,0.909091
1,140,public boolean isEmpty() {return first;},public virtual bool IsEmpty(){return first;},gpt-oss-120b,zero_shot_prompt,public bool IsEmpty()\n{\n return first;\n},False,0.817613,0.433661,0.909091
2,1434,public int lastLength() {return lastLength;},public virtual int LastLength(){return lastLen...,gpt-oss-120b,over_supervised_prompt,public int LastLength() { return lastLength; },False,0.817613,0.433661,0.909091
3,140,public boolean isEmpty() {return first;},public virtual bool IsEmpty(){return first;},gpt-oss-120b,over_supervised_prompt,public bool IsEmpty() { return first; },False,0.817613,0.433661,0.909091
4,5959,public String getScheme() {return scheme;},public string getScheme(){return scheme;},gpt-oss-120b,over_supervised_prompt,public string GetScheme() { return scheme; },False,0.707107,0.565726,1.000000
...,...,...,...,...,...,...,...,...,...,...
7995,3087,public List<String> call() throws GitAPIExcept...,public override IList<string> Call(){CheckCall...,qwen3-coder,over_supervised_prompt,public List<string> call() throws GitAPIExcept...,False,0.618705,0.392483,0.895238
7996,6229,public Note call() throws GitAPIException {che...,public override Note Call(){CheckCallable();Re...,qwen3-coder,zero_shot_prompt,public Note call() {\n checkCallable();\n ...,False,0.633182,0.485253,0.816327
7997,1439,"public ValueEval evaluate(int srcRowIndex, int...",public override ValueEval Evaluate(int srcRowI...,qwen3-coder,zero_shot_prompt,"public ValueEval evaluate(int srcRowIndex, int...",False,0.695638,0.391521,0.929293
7998,9920,public String toString(){StringBuilder buffer ...,public override String ToString(){StringBuilde...,qwen3-coder,zero_shot_prompt,public override string ToString()\n{\n Stri...,False,0.860008,0.444666,0.914286


# AI Judge

In [11]:
prompt_judge = """
You are a judge tasked with detecting potential cheating among students.

Task Description:
The assignment is to translate {src_lang} code into {tgt_lang}. You will be given:

1. One {src_lang} code snippet (the original source)
2. A {tgt_lang} translation from Student A
3. A {tgt_lang} translation from Student B

Your job is to determine whether Student B copied ideas from Student A.

Focus ONLY on elements in Student B’s solution that could reasonably have been
inferred from Student A’s answer but could NOT have been inferred from:
- the original {src_lang} code, or
- standard {src_lang}-to-{tgt_lang} translation knowledge.

Do NOT penalize common patterns, idiomatic translations, or stylistic similarities
that are typical for {src_lang}-to-{tgt_lang} conversions.

Evaluation Rubric:
(Score each category from 0–2, and explain the reasoning for that score.)

1. Derivability from {src_lang}
   0 → Fully inferable from {src_lang}
   1 → Slightly non-obvious but plausible
   2 → Not inferable from {src_lang}

2. Non-Obvious Design Choices
   0 → Standard / idiomatic
   1 → Questionable but explainable
   2 → Distinctive and unnecessary, matches Student A

3. Shared Errors or Quirks
   0 → None shared
   1 → One shared oddity
   2 → Multiple shared mistakes or quirks

4. Optional or Advanced Features
   0 → Expected usage
   1 → Similar optional usage
   2 → Same unnecessary advanced features

Scoring Interpretation:
0–3   → No evidence of copying
4–6   → Suspicious similarities
7–8   → Strong evidence Student B copied ideas from Student A

Final Rule:
Conclude copying ONLY if similarities are unlikely to have arisen independently.

You MUST:
- Assign a score (0–2) for each rubric category
- Provide a brief explanation justifying each score
- Compute the total score
- State the final conclusion based on the scoring interpretation

Original source {src_lang}:
{src}

{tgt_lang} translation from Student A:
{target}

{tgt_lang} translation from Student B:
{output}

You must answer with the following JSON format. Write only the JSON output in your response
{{
  "rubric": {{
    "derivability_from_source": {{
      "score": <0-2>,
      "explanation": "<why this score was given>"
    }},
    "non_obvious_design_choices": {{
      "score": <0-2>,
      "explanation": "<why this score was given>"
    }},
    "shared_errors_or_quirks": {{
      "score": <0-2>,
      "explanation": "<why this score was given>"
    }},
    "optional_or_advanced_features": {{
      "score": <0-2>,
      "explanation": "<why this score was given>"
    }}
  }},
  "total_score": <sum of all scores>,
}}
"""

In [12]:
df_judge = df[df['prompt_type']=='over_supervised_prompt'].drop_duplicates(['java_code','cs_code','model_output'])
len(df_judge)

3304

In [14]:
from litellm import completion
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def process_row(idx, row):
    try:
        formatted_prompt = prompt_judge.format(
            src=row['java_code'],
            target=row['cs_code'],
            output=row['model_output'],
            src_lang='Java',
            tgt_lang='C#'
        )

        response = completion(
            # model="openrouter/openai/gpt-oss-120b",
            # model="openrouter/openai/gpt-5-mini",
            model="openrouter/deepseek/deepseek-v3.2",
            messages=[{"role": "user", "content": formatted_prompt}],
            num_retries=5
        )

        return idx, response.choices[0].message.content

    except Exception as e:
        return idx, f"ERROR: {str(e)}"


df_judge['judge_deepseek-32'] = None

MAX_WORKERS = 10  # adjust based on rate limits

SAVE_INTERVAL = 20  # Save every 20 rows
counter = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [
        executor.submit(process_row, idx, row)
        for idx, row in df_judge.iterrows()
    ]

    for future in tqdm(as_completed(futures), total=len(futures)):
        idx, result = future.result()
        df_judge.at[idx, 'judge_deepseek-32'] = result

        counter += 1
        if counter % SAVE_INTERVAL == 0:
            df_judge.to_json("m1_metrics_checkpoint.jsonl", orient="records", lines=True)

print(f"✓ Complete! Processed all {len(df_judge)} rows")


100%|██████████| 3304/3304 [53:41<00:00,  1.03it/s]  


✓ Complete! Processed all 3304 rows


In [18]:
df_judge.to_json("m1_metrics.jsonl", orient="records", lines=True)

In [16]:
import re
import json
def extract_json(text: str):
    """
    Extracts the first valid JSON object from a string.
    Returns a Python dict or list.
    Raises ValueError if no valid JSON is found.
    """

    # 1. Try to extract from fenced code block ```json ... ```
    code_block = re.search(r"```json\s*(.*?)\s*```", text, re.DOTALL)
    if code_block:
        try:
            return json.loads(code_block.group(1))
        except json.JSONDecodeError:
            pass

    # 2. Fallback: find first {...} or [...]
    stack = []
    start = None

    for i, ch in enumerate(text):
        if ch in "{[":
            if not stack:
                start = i
            stack.append(ch)
        elif ch in "}]":
            if stack:
                stack.pop()
                if not stack and start is not None:
                    candidate = text[start:i + 1]
                    try:
                        return json.loads(candidate)
                    except json.JSONDecodeError:
                        start = None
    return None

In [ ]:
df_judge['judge_deepseek-32'] = df_judge['judge_deepseek-32'].apply(lambda x:extract_json(x))
df_judge['total_score_deepseek-32'] = df_judge['judge_deepseek-32'].apply(lambda x:x.get('total_score',-1) if x else 0)

In [24]:
df_judge['total_score_deepseek-32'].value_counts()

total_score_deepseek-32
0                                 2259
1                                  422
2                                  391
4                                   73
3                                   56
6                                   39
5                                   38
-1                                   7
8                                    5
7                                    4
0                                    4
                                     2
-4                                   1
[4-6, Suspicious similarities]       1
{}                                   1
(0)                                  1
Name: count, dtype: int64

In [27]:
from concurrent.futures import ThreadPoolExecutor, as_completed

# Rerun judge (multithreaded) on samples where the existing judge score is > 1
high_score_samples = df_judge[df_judge['total_score_deepseek-32'] != 0]

print(f"Found {len(high_score_samples)} samples with total_score_deepseek-32 > 1")

# Ensure the output column exists
if 'judge_rerun' not in df_judge.columns:
    df_judge['judge_rerun'] = None

MAX_WORKERS = 10  # adjust based on rate limits
SAVE_INTERVAL = 20  # Save every 20 rows


def rerun_row(idx, row):
    formatted_prompt = prompt_judge.format(
        src=row['java_code'],
        target=row['cs_code'],
        output=row['model_output'],
        src_lang='Java',
        tgt_lang='C#'
    )

    response = completion(
        # model="openrouter/deepseek/deepseek-v3.2",  # or change to another model if needed
        model="openrouter/openai/gpt-5-mini",
        messages=[{"role": "user", "content": formatted_prompt}],
        num_retries=5
    )

    return idx, response.choices[0].message.content


counter = 0
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [executor.submit(rerun_row, idx, row) for idx, row in high_score_samples.iterrows()]

    for future in tqdm(as_completed(futures), total=len(futures)):
        idx, result = future.result()
        df_judge.at[idx, 'judge_rerun'] = result

        counter += 1
        if counter % SAVE_INTERVAL == 0:
            df_judge.to_json("m1_metrics_checkpoint.jsonl", orient="records", lines=True)

print("Rerun complete!")

Found 1045 samples with total_score_deepseek-32 > 1


100%|██████████| 1045/1045 [30:45<00:00,  1.77s/it]

Rerun complete!


In [ ]:
df_judge['judge_rerun'] = df_judge['judge_rerun'].apply(
    lambda x: extract_json(x) if pd.notnull(x) else x
)
df_judge['total_score_gpt_5'] = df_judge['judge_rerun'].apply(lambda x:x.get('total_score',-1) if pd.notnull(x) else 0)

In [59]:
df_judge.loc[df_judge['total_score_deepseek-32']=='', 'total_score_deepseek-32'] = 0

In [66]:
df_judge['final_score'] = df_judge['total_score_gpt_5'].combine_first(df_judge['total_score_deepseek-32'])

/tmp/ipykernel_31767/1391665840.py:1: FutureWarning: The behavior of array concatenation with empty entries is deprecated. In a future version, this will no longer exclude empty items when determining the result dtype. To retain the old behavior, exclude the empty entries before the concat operation.
  df_judge['final_score'] = df_judge['total_score_gpt_5'].combine_first(df_judge['total_score_deepseek-32'])


In [68]:
df_judge['final_score'].value_counts()

final_score
0    3274
1      21
2       7
3       2
Name: count, dtype: int64

In [69]:
df_judge.to_json("m1_metrics.jsonl", orient="records", lines=True)

# Judge test

In [72]:
df_clean = df.drop_duplicates(['java_code','cs_code'])

In [ ]:
df_clean

/tmp/ipykernel_31767/555129692.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean.drop(columns=['judge_deepseek-32'],inplace=True)


In [ ]:
from litellm import completion
from concurrent.futures import ThreadPoolExecutor, as_completed
from tqdm import tqdm

def process_row(idx, row):
    try:
        formatted_prompt = prompt_judge.format(
            src=row['java_code'],
            target=row['cs_code'],
            output=row['cs_code'],
            src_lang='Java',
            tgt_lang='C#'
        )

        response = completion(
            # model="openrouter/deepseek/deepseek-v3.2",
            model="openrouter/openai/gpt-5-mini",
            messages=[{"role": "user", "content": formatted_prompt}],
            num_retries=5
        )

        return idx, response.choices[0].message.content

    except Exception as e:
        return idx, f"ERROR: {str(e)}"


df_clean['judge_deepseek-32'] = None

MAX_WORKERS = 10  # adjust based on rate limits

SAVE_INTERVAL = 20  # Save every 20 rows
counter = 0

with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
    futures = [
        executor.submit(process_row, idx, row)
        for idx, row in df_clean.iterrows()
    ]

    for future in tqdm(as_completed(futures), total=len(futures)):
        idx, result = future.result()
        df_clean.at[idx, 'judge_deepseek-32'] = result

        counter += 1
        if counter % SAVE_INTERVAL == 0:
            df_clean.to_json("m1_metrics_copy_checkpoint.jsonl", orient="records", lines=True)

print(f"✓ Complete! Processed all {len(df_clean)} rows")
df_clean.to_json("m1_metrics_copy.jsonl", orient="records", lines=True)


/tmp/ipykernel_31767/1246492033.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['judge_deepseek-32'] = None
100%|██████████| 1000/1000 [27:37<00:00,  1.66s/it]


✓ Complete! Processed all 1000 rows


In [77]:
df_clean['judge_deepseek-32'] = df_clean['judge_deepseek-32'].apply(lambda x:extract_json(x))
df_clean['total_score_deepseek-32'] = df_clean['judge_deepseek-32'].apply(lambda x:x.get('total_score',-1) if x else 0)

/tmp/ipykernel_31767/2498283635.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['judge_deepseek-32'] = df_clean['judge_deepseek-32'].apply(lambda x:extract_json(x))
/tmp/ipykernel_31767/2498283635.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_clean['total_score_deepseek-32'] = df_clean['judge_deepseek-32'].apply(lambda x:x.get('total_score',-1) if x else 0)


In [78]:
df_clean['total_score_deepseek-32'].value_counts()

total_score_deepseek-32
0    989
1     10
2      1
Name: count, dtype: int64